In [110]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [111]:
PROJECT_ROOT = Path.cwd().resolve()

RAW_DIR = PROJECT_ROOT / "Raw_Dataset"
PROCESSED_DIR = PROJECT_ROOT / "Processed_Dataset"
TABLE_DIR = PROJECT_ROOT / "Outputs" / "Tables"
FIGURE_DIR = PROJECT_ROOT / "Outputs" / "Figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "diabetic_data.csv"
IDS_PATH = RAW_DIR / "IDS_mapping.csv"

print(DATA_PATH)
print(IDS_PATH)

/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Raw_Dataset/diabetic_data.csv
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Raw_Dataset/IDS_mapping.csv


## Data Preping and Cleaning

In [112]:
df_raw = pd.read_csv(DATA_PATH, keep_default_na=False)
ids_mapping = pd.read_csv(IDS_PATH, keep_default_na=False)

df_raw["A1Cresult"].value_counts(dropna=False)

print("Raw diabetes data shape:", df_raw.shape)
print("IDS mapping shape:", ids_mapping.shape)

df_raw.head()

Raw diabetes data shape: (101766, 50)
IDS mapping shape: (67, 2)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [113]:
df_raw.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'payer_code',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

In [114]:
df = df_raw.copy()

df = df.replace("?", np.nan)

missing_summary = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .round(2)
    .reset_index()
)

missing_summary.columns = ["feature", "missing_percent"]
missing_summary.head(15)

,feature,missing_percent
0,weight,96.86
1,medical_specialty,49.08
2,payer_code,39.56
3,race,2.23
4,diag_3,1.40
5,diag_2,0.35
6,diag_1,0.02
7,encounter_id,0.00
8,tolazamide,0.00
9,glyburide,0.00


In [115]:
missing_summary.to_csv(TABLE_DIR / "missing_summary_raw_table1.csv", index=False)

In [116]:
row_log = []

def log_rows(step_name, data):
    row_log.append({
        "step": step_name,
        "rows": len(data),
        "columns": data.shape[1]
    })
    print(f"{step_name}: {data.shape}")
    
df_clean = df.copy()
log_rows("Raw data", df_clean)

Raw data: (101766, 50)


In [117]:
df_clean = df_clean.drop(columns=["weight", "payer_code"], errors="ignore")
log_rows("Dropped weight and payer_code", df_clean)

Dropped weight and payer_code: (101766, 48)


In [118]:
df_clean["medical_specialty"] = df_clean["medical_specialty"].fillna("Missing")
df_clean["race"] = df_clean["race"].fillna("Missing")

In [119]:
death_hospice_ids = [11, 13, 14, 19, 20, 21]

df_clean = df_clean[
    ~df_clean["discharge_disposition_id"].isin(death_hospice_ids)
].copy()

log_rows("Removed death/hospice discharge cases", df_clean)

Removed death/hospice discharge cases: (99343, 48)


In [120]:
df_clean = df_clean[df_clean["gender"] != "Unknown/Invalid"].copy()

log_rows("Removed Unknown/Invalid gender", df_clean)

Removed Unknown/Invalid gender: (99340, 48)


In [121]:
df_clean = (
    df_clean
    .sort_values(["patient_nbr", "encounter_id"])
    .drop_duplicates(subset="patient_nbr", keep="first")
    .copy()
)

log_rows("Kept first encounter per patient", df_clean)

Kept first encounter per patient: (69987, 48)


In [122]:
row_log_df = pd.DataFrame(row_log)
row_log_df

,step,rows,columns
0,Raw data,101766,50
1,Dropped weight and payer_code,101766,48
2,Removed death/hospice discharge cases,99343,48
3,Removed Unknown/Invalid gender,99340,48
4,Kept first encounter per patient,69987,48


In [123]:
row_log_df.to_csv(TABLE_DIR / "preprocessing_row_log.csv", index=False)

In [124]:
df_clean["readmitted_30"] = (df_clean["readmitted"] == "<30").astype(int)

df_clean["readmitted_30"].value_counts()

readmitted_30
0    63702
1     6285
Name: count, dtype: int64

In [125]:
readmission_rate = df_clean["readmitted_30"].mean() * 100
print(f"30-day readmission rate: {readmission_rate:.2f}%")

30-day readmission rate: 8.98%


## Create Hb1Ac Group

In [126]:
def make_hba1c_group(row):
    a1c = row["A1Cresult"]
    change = row["change"]

    if a1c == "None":
        return "No test was performed"
    elif a1c == ">8" and change == "Ch":
        return "High, medication changed"
    elif a1c == ">8" and change == "No":
        return "High, medication not changed"
    else:
        return "Normal result of the test"

df_clean["hba1c_group"] = df_clean.apply(make_hba1c_group, axis=1)

df_clean["hba1c_group"].value_counts()

hba1c_group
No test was performed           57141
Normal result of the test        6607
High, medication changed         4058
High, medication not changed     2181
Name: count, dtype: int64

In [127]:
def make_primary_diagnosis_group(code):
    if pd.isna(code):
        return "Other"

    code = str(code)

    if code.startswith("V") or code.startswith("E"):
        return "Other"

    try:
        code_num = float(code)
    except ValueError:
        return "Other"

    if (390 <= code_num <= 459) or code_num == 785:
        return "Circulatory"
    elif (460 <= code_num <= 519) or code_num == 786:
        return "Respiratory"
    elif (520 <= code_num <= 579) or code_num == 787:
        return "Digestive"
    elif int(code_num) == 250:
        return "Diabetes"
    elif 800 <= code_num <= 999:
        return "Injury"
    elif 710 <= code_num <= 739:
        return "Musculoskeletal"
    elif (580 <= code_num <= 629) or code_num == 788:
        return "Genitourinary"
    elif 140 <= code_num <= 239:
        return "Neoplasms"
    else:
        return "Other"

df_clean["primary_diagnosis"] = df_clean["diag_1"].apply(make_primary_diagnosis_group)

df_clean["primary_diagnosis"].value_counts()

primary_diagnosis
Circulatory        21389
Other              12134
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Name: count, dtype: int64

In [128]:
age_group_map = {
    "[0-10)": "<=30",
    "[10-20)": "<=30",
    "[20-30)": "<=30",
    "[30-40)": "30-60",
    "[40-50)": "30-60",
    "[50-60)": "30-60",
    "[60-70)": ">60",
    "[70-80)": ">60",
    "[80-90)": ">60",
    "[90-100)": ">60"
}

df_clean["age_group"] = df_clean["age"].map(age_group_map)

df_clean["age_group"].value_counts()

age_group
>60      46308
30-60    21871
<=30      1808
Name: count, dtype: int64

In [129]:
def make_discharge_group(x):
    if x == 1:
        return "Home"
    else:
        return "Other"

df_clean["discharge_group"] = df_clean["discharge_disposition_id"].apply(make_discharge_group)

def make_race_group(x):
    if x in ["AfricanAmerican", "Caucasian", "Missing"]:
        return x
    else:
        return "Other"

df_clean["race_group"] = df_clean["race"].apply(make_race_group)

def make_admission_source_group(x):
    if x == 7:
        return "Emergency room"
    elif x in [1, 2, 3]:
        return "Physician/clinic referral"
    else:
        return "Other"

df_clean["admission_source_group"] = df_clean["admission_source_id"].apply(make_admission_source_group)

def make_medical_specialty_group(x):
    if x == "InternalMedicine":
        return "Internal Medicine"
    elif x == "Cardiology":
        return "Cardiology"
    elif x in ["Surgery-General", "Surgery-Cardiovascular/Thoracic", 
               "Surgery-Neuro", "Surgery-Vascular", "Surgery-Plastic",
               "Surgery-Cardiovascular", "Surgery-Colon&Rectal",
               "Surgery-Pediatric", "Surgery-Maxillofacial",
               "Surgery-PlasticwithinHeadandNeck"]:
        return "Surgery"
    elif x in ["Family/GeneralPractice"]:
        return "Family/General Practice"
    elif x == "Missing":
        return "Missing"
    else:
        return "Other"

df_clean["medical_specialty_group"] = df_clean["medical_specialty"].apply(make_medical_specialty_group)

df_clean["medical_specialty_group"].value_counts()

medical_specialty_group
Missing                    33652
Other                      12915
Internal Medicine          10641
Family/General Practice     4978
Cardiology                  4207
Surgery                     3594
Name: count, dtype: int64

In [130]:
def table3_summary(data, column):
    summary = (
        data.groupby(column, dropna=False)
        .agg(
            encounters=("readmitted_30", "size"),
            readmitted=("readmitted_30", "sum"),
            readmission_rate=("readmitted_30", "mean")
        )
        .reset_index()
    )

    summary["population_percent"] = summary["encounters"] / len(data) * 100
    summary["readmission_rate"] = summary["readmission_rate"] * 100

    summary = summary[
        [column, "encounters", "population_percent", "readmitted", "readmission_rate"]
    ]

    summary["population_percent"] = summary["population_percent"].round(1)
    summary["readmission_rate"] = summary["readmission_rate"].round(1)

    return summary

In [131]:
hba1c_summary = table3_summary(df_clean, "hba1c_group")
hba1c_summary

,hba1c_group,encounters,population_percent,readmitted,readmission_rate
0,"High, medication changed",4058,5.8,348,8.6
1,"High, medication not changed",2181,3.1,161,7.4
2,No test was performed,57141,81.6,5206,9.1
3,Normal result of the test,6607,9.4,570,8.6


In [132]:
hba1c_summary.to_csv(TABLE_DIR / "table3_hba1c_summary.csv", index=False)

In [133]:
gender_summary = table3_summary(df_clean, "gender")
discharge_summary = table3_summary(df_clean, "discharge_group")
admission_summary = table3_summary(df_clean, "admission_source_group")
specialty_summary = table3_summary(df_clean, "medical_specialty_group")
diagnosis_summary = table3_summary(df_clean, "primary_diagnosis")
race_summary = table3_summary(df_clean, "race_group")
age_summary = table3_summary(df_clean, "age_group")

display(gender_summary)
display(discharge_summary)
display(admission_summary)
display(specialty_summary)
display(diagnosis_summary)
display(race_summary)
display(age_summary)

,gender,encounters,population_percent,readmitted,readmission_rate
0,Female,37239,53.2,3365,9.0
1,Male,32748,46.8,2920,8.9


,discharge_group,encounters,population_percent,readmitted,readmission_rate
0,Home,44320,63.3,3079,6.9
1,Other,25667,36.7,3206,12.5


,admission_source_group,encounters,population_percent,readmitted,readmission_rate
0,Emergency room,37271,53.3,3452,9.3
1,Other,9924,14.2,860,8.7
2,Physician/clinic referral,22792,32.6,1973,8.7


,medical_specialty_group,encounters,population_percent,readmitted,readmission_rate
0,Cardiology,4207,6.0,303,7.2
1,Family/General Practice,4978,7.1,485,9.7
2,Internal Medicine,10641,15.2,1039,9.8
3,Missing,33652,48.1,3110,9.2
4,Other,12915,18.5,1062,8.2
5,Surgery,3594,5.1,286,8.0


,primary_diagnosis,encounters,population_percent,readmitted,readmission_rate
0,Circulatory,21389,30.6,2070,9.7
1,Diabetes,5748,8.2,524,9.1
2,Digestive,6488,9.3,520,8.0
3,Genitourinary,3441,4.9,309,9.0
4,Injury,4694,6.7,507,10.8
5,Musculoskeletal,4064,5.8,341,8.4
6,Neoplasms,2538,3.6,230,9.1
7,Other,12134,17.3,1091,9.0
8,Respiratory,9491,13.6,693,7.3


,race_group,encounters,population_percent,readmitted,readmission_rate
0,AfricanAmerican,12627,18.0,1095,8.7
1,Caucasian,52305,74.7,4807,9.2
2,Missing,1917,2.7,141,7.4
3,Other,3138,4.5,242,7.7


,age_group,encounters,population_percent,readmitted,readmission_rate
0,30-60,21871,31.3,1574,7.2
1,<=30,1808,2.6,112,6.2
2,>60,46308,66.2,4599,9.9


In [134]:
gender_summary.to_csv(TABLE_DIR / "table3_gender_summary.csv", index=False)
discharge_summary.to_csv(TABLE_DIR / "table3_discharge_summary.csv", index=False)
admission_summary.to_csv(TABLE_DIR / "table3_admission_summary.csv", index=False)
specialty_summary.to_csv(TABLE_DIR / "table3_specialty_summary.csv", index=False)
diagnosis_summary.to_csv(TABLE_DIR / "table3_diagnosis_summary.csv", index=False)
race_summary.to_csv(TABLE_DIR / "table3_race_summary.csv", index=False)
age_summary.to_csv(TABLE_DIR / "table3_age_summary.csv", index=False)

In [135]:
df_clean.to_csv(PROCESSED_DIR / "diabetic_data_cleaned_stage1.csv", index=False)

print("Saved cleaned dataset to:")
print(PROCESSED_DIR / "diabetic_data_cleaned_stage1.csv")

Saved cleaned dataset to:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv


### Combine all summaries into a single table 3 style table

In [136]:
def table3_summary(data, column, variable_name, category_order=None):
    """
    Create one Table 3-style block for a categorical variable.

    Output columns:
    - Variable
    - Category
    - Number of encounters
    - % of population
    - Readmitted: number of encounters
    - Readmitted: % in group
    """

    temp = data.copy()

    if category_order is not None:
        temp[column] = pd.Categorical(
            temp[column],
            categories=category_order,
            ordered=True
        )

    summary = (
        temp.groupby(column, dropna=False, observed=False)
        .agg(
            number_of_encounters=("readmitted_30", "size"),
            readmitted_number=("readmitted_30", "sum"),
            readmitted_percent=("readmitted_30", "mean")
        )
        .reset_index()
    )

    summary["population_percent"] = summary["number_of_encounters"] / len(temp) * 100
    summary["readmitted_percent"] = summary["readmitted_percent"] * 100

    summary = summary.rename(columns={column: "Category"})

    summary.insert(0, "Variable", variable_name)

    summary = summary[
        [
            "Variable",
            "Category",
            "number_of_encounters",
            "population_percent",
            "readmitted_number",
            "readmitted_percent"
        ]
    ]

    summary = summary.rename(columns={
        "number_of_encounters": "Number of encounters",
        "population_percent": "% of population",
        "readmitted_number": "Readmitted: Number of encounters",
        "readmitted_percent": "Readmitted: % in group"
    })

    summary["% of population"] = summary["% of population"].round(1)
    summary["Readmitted: % in group"] = summary["Readmitted: % in group"].round(1)

    return summary

In [137]:
hba1c_order = [
    "No test was performed",
    "High, medication changed",
    "High, medication not changed",
    "Normal result of the test"
]

gender_order = [
    "Female",
    "Male"
]

discharge_order = [
    "Home",
    "Other"
]

admission_order = [
    "Emergency room",
    "Physician/clinic referral",
    "Other"
]

specialty_order = [
    "Internal Medicine",
    "Cardiology",
    "Surgery",
    "Family/General Practice",
    "Missing",
    "Other"
]

diagnosis_order = [
    "Circulatory",
    "Diabetes",
    "Respiratory",
    "Digestive",
    "Injury",
    "Musculoskeletal",
    "Genitourinary",
    "Neoplasms",
    "Other"
]

race_order = [
    "AfricanAmerican",
    "Caucasian",
    "Other",
    "Missing"
]

age_order = [
    "<=30",
    "30-60",
    ">60"
]

In [138]:
table3_hba1c = table3_summary(
    df_clean,
    "hba1c_group",
    "HbA1c",
    hba1c_order
)

table3_gender = table3_summary(
    df_clean,
    "gender",
    "Gender",
    gender_order
)

table3_discharge = table3_summary(
    df_clean,
    "discharge_group",
    "Discharge disposition",
    discharge_order
)

table3_admission = table3_summary(
    df_clean,
    "admission_source_group",
    "Admission source",
    admission_order
)

table3_specialty = table3_summary(
    df_clean,
    "medical_specialty_group",
    "Specialty of the admitting physician",
    specialty_order
)

table3_diagnosis = table3_summary(
    df_clean,
    "primary_diagnosis",
    "Primary diagnosis",
    diagnosis_order
)

table3_race = table3_summary(
    df_clean,
    "race_group",
    "Race",
    race_order
)

table3_age = table3_summary(
    df_clean,
    "age_group",
    "Age",
    age_order
)

In [139]:
table3_combined = pd.concat(
    [
        table3_hba1c,
        table3_gender,
        table3_discharge,
        table3_admission,
        table3_specialty,
        table3_diagnosis,
        table3_race,
        table3_age
    ],
    ignore_index=True
)

table3_combined

,Variable,Category,Number of encounters,% of population,Readmitted: Number of encounters,Readmitted: % in group
0,HbA1c,No test was performed,57141,81.6,5206,9.1
1,HbA1c,"High, medication changed",4058,5.8,348,8.6
2,HbA1c,"High, medication not changed",2181,3.1,161,7.4
3,HbA1c,Normal result of the test,6607,9.4,570,8.6
4,Gender,Female,37239,53.2,3365,9.0
5,Gender,Male,32748,46.8,2920,8.9
6,Discharge disposition,Home,44320,63.3,3079,6.9
7,Discharge disposition,Other,25667,36.7,3206,12.5
8,Admission source,Emergency room,37271,53.3,3452,9.3
9,Admission source,Physician/clinic referral,22792,32.6,1973,8.7


In [140]:
table3_display = table3_combined.copy()

table3_display["Variable"] = table3_display["Variable"].mask(
    table3_display["Variable"].duplicated(),
    ""
)

table3_display

,Variable,Category,Number of encounters,% of population,Readmitted: Number of encounters,Readmitted: % in group
0,HbA1c,No test was performed,57141,81.6,5206,9.1
1,,"High, medication changed",4058,5.8,348,8.6
2,,"High, medication not changed",2181,3.1,161,7.4
3,,Normal result of the test,6607,9.4,570,8.6
4,Gender,Female,37239,53.2,3365,9.0
5,,Male,32748,46.8,2920,8.9
6,Discharge disposition,Home,44320,63.3,3079,6.9
7,,Other,25667,36.7,3206,12.5
8,Admission source,Emergency room,37271,53.3,3452,9.3
9,,Physician/clinic referral,22792,32.6,1973,8.7


In [141]:
age_midpoint_map = {
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
}

df_clean["age_midpoint"] = df_clean["age"].map(age_midpoint_map)

In [142]:
def continuous_summary(data, column, variable_name, category_label):
    values = data[column].dropna()

    return pd.DataFrame({
        "Variable": [variable_name],
        "Category": [category_label],
        "Mean": [values.mean().round(1)],
        "Median": [values.median().round(1)],
        "1st Qu.": [values.quantile(0.25).round(1)],
        "3rd Qu.": [values.quantile(0.75).round(1)]
    })

In [143]:
age_numeric_summary = continuous_summary(
    df_clean,
    "age_midpoint",
    "Age (numeric)",
    "Age in years"
)

time_hospital_summary = continuous_summary(
    df_clean,
    "time_in_hospital",
    "Time in hospital",
    "Days between admission and discharge"
)

table3_continuous = pd.concat(
    [age_numeric_summary, time_hospital_summary],
    ignore_index=True
)

table3_continuous

,Variable,Category,Mean,Median,1st Qu.,3rd Qu.
0,Age (numeric),Age in years,65.4,65.0,55.0,75.0
1,Time in hospital,Days between admission and discharge,4.3,3.0,2.0,6.0


In [144]:
table3_combined.to_csv(
    TABLE_DIR / "table3_categorical_combined.csv",
    index=False
)

table3_display.to_csv(
    TABLE_DIR / "table3_categorical_display_style.csv",
    index=False
)

table3_continuous.to_csv(
    TABLE_DIR / "table3_continuous_summary.csv",
    index=False
)

In [145]:
display(table3_display)
display(table3_continuous)

,Variable,Category,Number of encounters,% of population,Readmitted: Number of encounters,Readmitted: % in group
0,HbA1c,No test was performed,57141,81.6,5206,9.1
1,,"High, medication changed",4058,5.8,348,8.6
2,,"High, medication not changed",2181,3.1,161,7.4
3,,Normal result of the test,6607,9.4,570,8.6
4,Gender,Female,37239,53.2,3365,9.0
5,,Male,32748,46.8,2920,8.9
6,Discharge disposition,Home,44320,63.3,3079,6.9
7,,Other,25667,36.7,3206,12.5
8,Admission source,Emergency room,37271,53.3,3452,9.3
9,,Physician/clinic referral,22792,32.6,1973,8.7


,Variable,Category,Mean,Median,1st Qu.,3rd Qu.
0,Age (numeric),Age in years,65.4,65.0,55.0,75.0
1,Time in hospital,Days between admission and discharge,4.3,3.0,2.0,6.0
